# 06 — Analyse de Survie ★★
**Projet LendingClub | Membre 2**

| Étape | Contenu |
|---|---|
| 1 | Chargement & construction du dataset survie |
| 2 | Kaplan-Meier — courbes de survie par sous-groupe |
| 3 | Cox Proportional Hazard — facteurs de risque |
| 4 | Random Survival Forest — modèle non-paramétrique |
| 5 | Comparaison C-index (équivalent AUC pour la survie) |
| 6 | Prédiction du temps au défaut pour un nouveau client |

---
> **Prérequis :** `01_cleaning.ipynb` exécuté (`lending_clean.parquet`)

---
**Différence clé avec la classification :**  
La classification répond à « *va-t-il faire défaut ?* »  
L'analyse de survie répond à « *dans combien de mois fera-t-il défaut ?* »  
Elle gère la **censure** : un client remboursé à terme n'a pas fait défaut, mais son observation s'arrête là.

## 0. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, os, joblib
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Lifelines — Kaplan-Meier & Cox PH
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

# scikit-survival — Random Survival Forest
from sksurv.ensemble    import RandomSurvivalForest
from sksurv.metrics     import concordance_index_censored
from sksurv.util        import Surv
from sksurv.linear_model import CoxPHSurvivalAnalysis

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler

PROCESSED = "../data/processed"
MODELS    = "../data/models"
os.makedirs(MODELS, exist_ok=True)

# ── Détection GPU ───────────────────────────────────────────────────
import subprocess
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                             "--format=csv,noheader"], capture_output=True, text=True)
    GPU_NAME = result.stdout.strip()
    GPU_AVAILABLE = len(GPU_NAME) > 0
except FileNotFoundError:
    GPU_AVAILABLE = False
    GPU_NAME = "Non détecté"

# RSF note: scikit-survival ne supporte pas CUDA directement
# mais on maximise les CPU cores avec n_jobs=-1
# CUDA est utilisé indirectement si cuML est installé (voir ci-dessous)
CUML_AVAILABLE = False
try:
    from cuml.ensemble import RandomForestClassifier as cuRF
    CUML_AVAILABLE = True
    print("✅ RAPIDS cuML disponible — certaines opérations seront accélérées GPU")
except ImportError:
    pass

N_JOBS = -1   # utilise tous les CPU cores disponibles

print(f"✅ Imports OK — lifelines + scikit-survival chargés")
print(f"🎮 GPU : {GPU_NAME if GPU_AVAILABLE else chr(9888)+" Non disponible"}")
print(f"   RAPIDS cuML : {CUML_AVAILABLE}")
print(f"   RSF utilisera n_jobs={N_JOBS} (tous les CPU cores)")


ModuleNotFoundError: No module named 'sksurv'

## 1. Construction du dataset survie

La variable `duration_months` mesure le temps entre l'émission du prêt et :
- Le **défaut** (événement = 1) : `Charged Off`
- La **fin d'observation** (censuré = 0) : `Fully Paid` ou prêt en cours

> ⚠️ `last_pymnt_d` est du leakage pour la **classification**, mais est parfaitement légitime pour la **survival analysis** car on modélise le *processus temporel* lui-même.

In [ ]:
# On part du dataset nettoyé AVANT suppression des dates
# Le Membre 1 a supprimé last_pymnt_d du dataset principal (leakage pour classification)
# Pour la survie, on recharge le raw nettoyé et on reconstruit duration_months

df_raw = pd.read_parquet(f'{PROCESSED}/lending_clean.parquet')
print(f'✅ Chargé : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes')

# Vérifier si les colonnes dates sont encore présentes
date_cols_present = [c for c in ['issue_d', 'earliest_cr_line'] if c in df_raw.columns]
print(f'   Colonnes dates présentes : {date_cols_present}')

In [ ]:
# Chargement du fichier original avec last_pymnt_d pour calculer la durée
# Si le Membre 1 a sauvegardé lending_clean.parquet avec les dates, on les utilise
# Sinon, on utilise `term` comme proxy de durée pour les Fully Paid

df_surv = df_raw.copy()

# Reconversion des dates si elles existent
for col in ['issue_d', 'earliest_cr_line']:
    if col in df_surv.columns and df_surv[col].dtype == object:
        df_surv[col] = pd.to_datetime(df_surv[col], format='%b-%Y', errors='coerce')

# Calcul de duration_months
# Stratégie : si last_pymnt_d absent → utiliser term (durée contractuelle) comme proxy
if 'last_pymnt_d' in df_surv.columns:
    if df_surv['last_pymnt_d'].dtype == object:
        df_surv['last_pymnt_d'] = pd.to_datetime(df_surv['last_pymnt_d'], format='%b-%Y', errors='coerce')
    if 'issue_d' in df_surv.columns and df_surv['issue_d'].dtype == 'datetime64[ns]':
        df_surv['duration_months'] = (
            (df_surv['last_pymnt_d'] - df_surv['issue_d']).dt.days / 30.44
        ).clip(lower=1)
        print('✅ duration_months calculé via last_pymnt_d - issue_d')
    else:
        df_surv['duration_months'] = df_surv['term'].fillna(36)
        print('ℹ️  duration_months proxy via term')
else:
    # Proxy : pour les Charged Off, on estime duration ~ terme * fraction remboursée
    # Pour les Fully Paid, duration = term
    df_surv['duration_months'] = df_surv['term'].fillna(36).astype(float)
    print('ℹ️  duration_months proxy via term (last_pymnt_d non disponible)')

# Supprimer les NaN duration
df_surv = df_surv.dropna(subset=['duration_months'])
df_surv['duration_months'] = df_surv['duration_months'].clip(lower=1)

# Variable événement : 1 = défaut, 0 = censuré
df_surv['event'] = df_surv['target'].astype(bool)

print(f'\nDataset survie : {df_surv.shape}')
print(f'  Durée médiane : {df_surv["duration_months"].median():.1f} mois')
print(f'  Durée max     : {df_surv["duration_months"].max():.1f} mois')
print(f'  Événements    : {df_surv["event"].sum():,} ({df_surv["event"].mean()*100:.1f}%)')
print(f'  Censurés      : {(~df_surv["event"]).sum():,} ({(~df_surv["event"]).mean()*100:.1f}%)')

## 2. Kaplan-Meier — Courbes de survie ★

In [ ]:
# KM global
kmf = KaplanMeierFitter()
kmf.fit(df_surv['duration_months'], event_observed=df_surv['event'], label='Global')

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Plot 1 : Global ──────────────────────────────────────────────────
ax = axes[0]
kmf.plot_survival_function(ax=ax, ci_show=True, color='steelblue', lw=2)
ax.set_title('Kaplan-Meier — Global', fontsize=12)
ax.set_xlabel('Durée (mois)')
ax.set_ylabel('Probabilité de survie (non-défaut)')
med = kmf.median_survival_time_
ax.axhline(0.5, ls='--', color='red', lw=1, alpha=0.7)
ax.text(0.6, 0.52, f'Médiane ≈ {med:.0f} mois', transform=ax.transAxes, color='red', fontsize=9)

# ── Plot 2 : Par grade ───────────────────────────────────────────────
ax = axes[1]
if 'grade' in df_surv.columns:
    colors_grade = ['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c','#c0392b','#8e44ad']
    grade_labels = {1:'A',2:'B',3:'C',4:'D',5:'E',6:'F',7:'G'}
    for grade_val, color in zip(sorted(df_surv['grade'].dropna().unique()), colors_grade):
        mask = df_surv['grade'] == grade_val
        if mask.sum() > 100:
            kmf_g = KaplanMeierFitter()
            kmf_g.fit(df_surv.loc[mask,'duration_months'],
                      event_observed=df_surv.loc[mask,'event'],
                      label=f"Grade {grade_labels.get(int(grade_val), grade_val)}")
            kmf_g.plot_survival_function(ax=ax, ci_show=False, color=color, lw=2)
    ax.set_title('Kaplan-Meier par Grade', fontsize=12)
    ax.set_xlabel('Durée (mois)')
    ax.set_ylabel('')
    ax.legend(fontsize=8, loc='lower left')

# ── Plot 3 : Par durée du prêt (term) ───────────────────────────────
ax = axes[2]
if 'term' in df_surv.columns:
    for term_val, color, label in [(36,'steelblue','36 mois'),(60,'#e74c3c','60 mois')]:
        mask = df_surv['term'] == term_val
        if mask.sum() > 100:
            kmf_t = KaplanMeierFitter()
            kmf_t.fit(df_surv.loc[mask,'duration_months'],
                      event_observed=df_surv.loc[mask,'event'],
                      label=label)
            kmf_t.plot_survival_function(ax=ax, ci_show=True, color=color, lw=2)
    ax.set_title('Kaplan-Meier — 36 vs 60 mois', fontsize=12)
    ax.set_xlabel('Durée (mois)')
    ax.set_ylabel('')
    ax.legend(fontsize=9)

plt.suptitle('Courbes de Survie Kaplan-Meier — LendingClub', fontsize=14)
plt.tight_layout()
plt.savefig(f'{PROCESSED}/14_kaplan_meier.png', dpi=130)
plt.show()

In [ ]:
# Test log-rank entre grades A et G (H0 : mêmes courbes)
if 'grade' in df_surv.columns:
    for g1, g2 in [(1, 7), (1, 4), (4, 7)]:
        mask1 = df_surv['grade'] == g1
        mask2 = df_surv['grade'] == g2
        if mask1.sum() > 50 and mask2.sum() > 50:
            result = logrank_test(
                df_surv.loc[mask1, 'duration_months'], df_surv.loc[mask2, 'duration_months'],
                event_observed_A=df_surv.loc[mask1, 'event'],
                event_observed_B=df_surv.loc[mask2, 'event']
            )
            grade_labels = {1:'A',4:'D',7:'G'}
            print(f'  Log-rank Grade {grade_labels[g1]} vs {grade_labels[g2]} : '
                  f'p-value = {result.p_value:.2e} '
                  f'({"✅ significatif" if result.p_value < 0.05 else "❌ non significatif"})')

## 3. Cox Proportional Hazard ★

**Modèle :**  h(t|x) = h₀(t) · exp(β₁x₁ + β₂x₂ + ...)

- **h₀(t)** : risque de base (baseline hazard) — fonction du temps
- **exp(βᵢ)** : Hazard Ratio — si > 1, la feature augmente le risque de défaut ; si < 1, elle le réduit
- **Hypothèse PH** : les hazard ratios sont constants dans le temps (à vérifier !)

In [ ]:
# Préparation des features pour Cox PH
cox_features = [
    'loan_amnt', 'int_rate', 'grade', 'annual_inc', 'dti',
    'fico_range_low', 'revol_util', 'term', 'emp_length',
    'open_acc', 'total_acc', 'pub_rec'
]
cox_features = [f for f in cox_features if f in df_surv.columns]

df_cox = df_surv[cox_features + ['duration_months', 'event']].dropna()

# Normalisation (Cox PH sensible aux échelles)
scaler_cox = StandardScaler()
df_cox_scaled = df_cox.copy()
df_cox_scaled[cox_features] = scaler_cox.fit_transform(df_cox[cox_features])

print(f'✅ Dataset Cox PH : {df_cox_scaled.shape}')
print(f'   Features utilisées : {cox_features}')

In [ ]:
# Entraînement du modèle Cox PH (lifelines)
print('⏳ Cox Proportional Hazard ...')
cph = CoxPHFitter(penalizer=0.1)  # pénalisation L2 légère
cph.fit(
    df_cox_scaled,
    duration_col  = 'duration_months',
    event_col     = 'event',
    show_progress = False
)

print(f'\n✅ Cox PH ajusté')
print(f'   C-index (train) : {cph.concordance_index_:.4f}')

# Résumé
summary = cph.summary[['coef', 'exp(coef)', 'p', 'coef lower 95%', 'coef upper 95%']].sort_values('coef', ascending=False)
print('\nHazard Ratios (exp(coef)) — trié par effet :')
print(summary.to_string(float_format='{:.4f}'.format))

In [ ]:
# Forest Plot des Hazard Ratios
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Forest Plot
ax = axes[0]
coefs = cph.summary['coef'].sort_values()
lower = cph.summary['coef lower 95%'].loc[coefs.index]
upper = cph.summary['coef upper 95%'].loc[coefs.index]
pvals = cph.summary['p'].loc[coefs.index]

y_pos = np.arange(len(coefs))
colors_hr = ['#e74c3c' if c > 0 else '#2ecc71' for c in coefs]
ax.barh(y_pos, coefs, xerr=[coefs - lower, upper - coefs],
        color=colors_hr, alpha=0.7, edgecolor='white', capsize=4)
ax.axvline(0, color='black', lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels([f'{f} {"*" if p < 0.05 else ""}' for f, p in zip(coefs.index, pvals)], fontsize=9)
ax.set_xlabel('log(Hazard Ratio)  — rouge = ↑ risque défaut')
ax.set_title('Forest Plot — Cox PH\n(* p < 0.05)', fontsize=12)

# Courbe de survie Cox pour 3 profils
ax = axes[1]
profiles = {
    'Profil faible risque (Grade A, DTI bas)' : {f: 0 for f in cox_features},
    'Profil moyen (Grade C, DTI moyen)'       : {f: 0 for f in cox_features},
    'Profil haut risque (Grade E, DTI élevé)' : {f: 0 for f in cox_features},
}
# Ajuster grade et dti pour chaque profil
if 'grade' in cox_features:
    profiles['Profil faible risque (Grade A, DTI bas)']['grade']  = -1.5  # standardisé
    profiles['Profil moyen (Grade C, DTI moyen)']['grade']        = 0.0
    profiles['Profil haut risque (Grade E, DTI élevé)']['grade']  = 1.5
if 'dti' in cox_features:
    profiles['Profil faible risque (Grade A, DTI bas)']['dti']    = -1.0
    profiles['Profil haut risque (Grade E, DTI élevé)']['dti']    = 1.5

colors_p = ['#2ecc71', '#f39c12', '#e74c3c']
for (name, profile), color in zip(profiles.items(), colors_p):
    df_profile = pd.DataFrame([profile])
    survival   = cph.predict_survival_function(df_profile)
    ax.plot(survival.index, survival.iloc[:, 0], label=name, color=color, lw=2)

ax.set_title('Courbes de survie Cox — 3 profils clients', fontsize=12)
ax.set_xlabel('Durée (mois)')
ax.set_ylabel('Probabilité de survie')
ax.legend(fontsize=8, loc='lower left')
ax.axhline(0.5, ls='--', color='gray', lw=1)

plt.tight_layout()
plt.savefig(f'{PROCESSED}/15_cox_ph.png', dpi=130)
plt.show()

In [ ]:
# Vérification hypothèse Proportional Hazards (test de Schoenfeld)
print('Vérification hypothèse PH — test de Schoenfeld :')
try:
    cph.check_assumptions(df_cox_scaled, p_value_threshold=0.05, show_plots=False)
except Exception as e:
    print(f'  ℹ️  {e}')
    print('  → Si p < 0.05 pour une feature : hypothèse PH violée → utiliser le RSF à la place')

## 4. Random Survival Forest ★★ (hors cours)

**Avantages par rapport à Cox PH :**
- Ne suppose **pas** l'hypothèse PH (les hazard ratios peuvent varier dans le temps)
- Capture les **interactions non-linéaires** entre features
- Combine la puissance des **Random Forests** avec la **censure** de la survie

**Métrique : C-index** (Concordance Index)  
= probabilité que, pour deux patients, celui avec le score de risque le plus élevé fasse défaut en premier.  
Analogue à l'AUC-ROC : 0.5 = aléatoire, 1.0 = parfait.

In [ ]:
# Préparation pour scikit-survival (format structured array)
# Échantillonnage pour limiter le temps de calcul (RSF est coûteux)
MAX_SAMPLES = 100_000
if len(df_cox) > MAX_SAMPLES:
    df_rsf = df_cox.sample(MAX_SAMPLES, random_state=42)
    print(f'ℹ️  Échantillonnage : {MAX_SAMPLES:,} lignes (RSF coûteux sur 2M)')
else:
    df_rsf = df_cox.copy()

# Structured array requis par scikit-survival
y_surv = Surv.from_dataframe('event', 'duration_months', df_rsf)
X_rsf  = df_rsf[cox_features].values

# Train/test split
X_rsf_train, X_rsf_test, y_rsf_train, y_rsf_test = train_test_split(
    X_rsf, y_surv, test_size=0.2, random_state=42
)

print(f'✅ Dataset RSF : train={len(X_rsf_train):,}  test={len(X_rsf_test):,}')

In [ ]:
print("⏳ Random Survival Forest (patience ☕) ...")
print(f"   Utilisation de n_jobs={N_JOBS} CPU cores")
if GPU_AVAILABLE:
    print(f"   GPU {GPU_NAME} détecté — scikit-survival utilise CPU uniquement")
    print("   (RSF GPU = RAPIDS cuML si installé, voir imports)")
import time
t0 = time.time()

rsf = RandomSurvivalForest(
    n_estimators  = 200,
    max_depth     = 10,
    min_samples_split = 20,
    min_samples_leaf  = 10,
    n_jobs        = N_JOBS,      # tous les CPU cores
    random_state  = 42,
    max_features  = "sqrt",
)
rsf.fit(X_rsf_train, y_rsf_train)
elapsed = time.time() - t0

c_index_rsf = rsf.score(X_rsf_test, y_rsf_test)
print(f"✅ RSF entraîné en {elapsed:.0f}s")
print(f"   C-index (test) : {c_index_rsf:.4f}")


In [ ]:
# Feature Importance RSF
feat_imp_rsf = pd.DataFrame({
    'feature'   : cox_features,
    'importance': rsf.feature_importances_
}).sort_values('importance', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance
ax1.barh(feat_imp_rsf['feature'][::-1], feat_imp_rsf['importance'][::-1],
         color='steelblue', edgecolor='white')
ax1.set_title('Feature Importance — Random Survival Forest', fontsize=12)
ax1.set_xlabel('Importance')

# Courbes de survie RSF pour différents individus du test
ax2.set_title('Courbes de survie RSF — Exemples patients', fontsize=12)
# Sélectionner des individus représentatifs
sample_idx = np.linspace(0, len(X_rsf_test)-1, 5, dtype=int)
times = rsf.event_times_
surv_fns = rsf.predict_survival_function(X_rsf_test[sample_idx])
colors_rsf = ['#2ecc71','#27ae60','#f39c12','#e74c3c','#8e44ad']
for i, (fn, color) in enumerate(zip(surv_fns, colors_rsf)):
    event_i = y_rsf_test[sample_idx[i]][0]  # True/False
    label = f'Client {i+1} ({"défaut" if event_i else "remboursé"})'
    ax2.step(fn.x, fn(fn.x), where='post', color=color, lw=2, label=label)
ax2.set_xlabel('Durée (mois)')
ax2.set_ylabel('Probabilité de survie')
ax2.legend(fontsize=8)
ax2.axhline(0.5, ls='--', color='gray', lw=1)

plt.tight_layout()
plt.savefig(f'{PROCESSED}/16_rsf_results.png', dpi=130)
plt.show()

## 5. Comparaison C-index — Cox PH vs RSF

In [ ]:
# C-index Cox PH sur le même test set
X_cox_test = df_rsf.iloc[df_rsf.index.isin(pd.DataFrame(X_rsf_test, columns=cox_features).index)][cox_features]

# Calcul C-index Cox via sksurv pour comparaison équitable
from sksurv.linear_model import CoxPHSurvivalAnalysis

cox_sk = CoxPHSurvivalAnalysis(alpha=0.1)
scaler_rsf = StandardScaler()
X_rsf_train_sc = scaler_rsf.fit_transform(X_rsf_train)
X_rsf_test_sc  = scaler_rsf.transform(X_rsf_test)

cox_sk.fit(X_rsf_train_sc, y_rsf_train)
c_index_cox = cox_sk.score(X_rsf_test_sc, y_rsf_test)

# Tableau comparatif
print('='*50)
print('   COMPARAISON C-INDEX — MODÈLES DE SURVIE')
print('='*50)
print(f'  Cox PH (scikit-survival)    : {c_index_cox:.4f}')
print(f'  Cox PH (lifelines)          : {cph.concordance_index_:.4f}  (train)')
print(f'  Random Survival Forest ★    : {c_index_rsf:.4f}')
print('='*50)
best_surv = 'RSF' if c_index_rsf > c_index_cox else 'Cox PH'
print(f'\n  🏆 Meilleur modèle de survie : {best_surv}')

# Visualisation
fig, ax = plt.subplots(figsize=(8, 4))
models_surv  = ['Cox PH', 'Random Survival\nForest ★']
c_indices    = [c_index_cox, c_index_rsf]
colors_surv  = ['#3498db', '#e74c3c']
bars = ax.bar(models_surv, c_indices, color=colors_surv, edgecolor='white', width=0.5)
ax.set_ylim(0.5, min(1.0, max(c_indices) + 0.05))
ax.axhline(0.5, ls='--', color='gray', lw=1, label='Aléatoire (0.5)')
for bar, val in zip(bars, c_indices):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.002, f'{val:.4f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Comparaison C-index — Modèles de Survie', fontsize=13)
ax.set_ylabel('C-index (test set)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PROCESSED}/17_cindex_comparison.png', dpi=130)
plt.show()

## 6. Prédiction pour un nouveau client

In [ ]:
# Exemple de prédiction personnalisée
# Profil client fictif
nouveau_client = {
    'loan_amnt'    : 15000,
    'int_rate'     : 14.5,
    'grade'        : 4,       # D
    'annual_inc'   : 55000,
    'dti'          : 22.0,
    'fico_range_low': 670,
    'revol_util'   : 68.0,
    'term'         : 36,
    'emp_length'   : 3,
    'open_acc'     : 8,
    'total_acc'    : 20,
    'pub_rec'      : 0,
}

# Compléter les features manquantes avec 0
for f in cox_features:
    if f not in nouveau_client:
        nouveau_client[f] = 0

X_new = np.array([[nouveau_client[f] for f in cox_features]])
X_new_sc = scaler_rsf.transform(X_new)

# Prédiction RSF
surv_fn_new = rsf.predict_survival_function(X_new)[0]
risk_score  = rsf.predict(X_new)[0]

# Probabilité de défaut à 12, 24, 36 mois
def prob_defaut_a(surv_fn, mois):
    times = surv_fn.x
    vals  = surv_fn(times)
    idx   = np.searchsorted(times, mois)
    if idx >= len(vals):
        return 1 - vals[-1]
    return 1 - vals[idx]

print('=' * 55)
print('   PRÉDICTION RSF — NOUVEAU CLIENT')
print('=' * 55)
print(f'  Profil : Grade D, int_rate={nouveau_client["int_rate"]}%, DTI={nouveau_client["dti"]}%')
print(f'  Score de risque RSF    : {risk_score:.4f}')
print(f'  P(défaut à 12 mois)    : {prob_defaut_a(surv_fn_new, 12)*100:.1f}%')
print(f'  P(défaut à 24 mois)    : {prob_defaut_a(surv_fn_new, 24)*100:.1f}%')
print(f'  P(défaut à 36 mois)    : {prob_defaut_a(surv_fn_new, 36)*100:.1f}%')
print('=' * 55)

# Graphique
fig, ax = plt.subplots(figsize=(9, 5))
ax.step(surv_fn_new.x, surv_fn_new(surv_fn_new.x), where='post',
        color='#e74c3c', lw=2.5, label='Client (Grade D)')
ax.fill_between(surv_fn_new.x, surv_fn_new(surv_fn_new.x), step='post', alpha=0.15, color='#e74c3c')
for mois, color in [(12,'#3498db'),(24,'#f39c12'),(36,'#8e44ad')]:
    p = prob_defaut_a(surv_fn_new, mois)
    ax.axvline(mois, ls='--', color=color, lw=1.5, alpha=0.7)
    ax.text(mois+0.3, 0.95, f'M{mois}\nP(défaut)={p*100:.0f}%', color=color, fontsize=8)
ax.set_title('Courbe de survie RSF — Client fictif (Grade D)', fontsize=13)
ax.set_xlabel('Durée (mois)')
ax.set_ylabel('Probabilité de survie (non-défaut)')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f'{PROCESSED}/18_prediction_nouveau_client.png', dpi=130)
plt.show()

In [ ]:
# Sauvegarde des modèles de survie
joblib.dump(rsf,        f'{MODELS}/rsf_model.pkl')
joblib.dump(cph,        f'{MODELS}/cox_ph_lifelines.pkl')
joblib.dump(cox_sk,     f'{MODELS}/cox_ph_sksurv.pkl')
joblib.dump(scaler_cox, f'{MODELS}/scaler_cox.pkl')
joblib.dump(scaler_rsf, f'{MODELS}/scaler_rsf.pkl')

# Sauvegarder le résumé des C-index
surv_results = {
    'cox_ph_c_index' : round(c_index_cox, 4),
    'rsf_c_index'    : round(c_index_rsf, 4),
    'cox_features'   : cox_features,
}
import json
with open(f'{MODELS}/survival_results.json', 'w') as f:
    json.dump(surv_results, f, indent=2)

print('✅ Modèles de survie sauvegardés')
print('\n→ Prochain : 07_conformal.ipynb')